# Hindi Model — resume 30000 -> 61036 steps (warm start)

Before running: upload `model_final.pt` from your 30000-step run as a Kaggle Dataset
(named e.g. `hindi-ckpt-30k`) and attach it via **Add Input**. Replace
`<your-dataset-name>` below with the actual mounted folder name shown in the
sidebar.

Also rotate the GitHub PAT and HF token below before running — put them in
Kaggle Secrets (Add-ons -> Secrets), not literals.

In [1]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

import os
os.environ["HF_TOKEN"] = HF_TOKEN

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-ManyaJain111.git"

# FIX: `git clone` errored with "destination path 'repo' already exists" on
# every re-run in the same session. Pull instead of cloning when it's already there.
if os.path.isdir("repo/.git"):
    print("[repo] already cloned -> pulling latest phase2 branch")
    !git -C repo checkout phase2
    !git -C repo pull origin phase2
else:
    !git clone -b phase2 {REPO_URL} repo


Cloning into 'repo'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (351/351), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 351 (delta 161), reused 205 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (351/351), 19.50 MiB | 23.69 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [2]:
!pip install -q huggingface_hub datasets sacrebleu rouge_score seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.1 MB/s eta 0:00:00


## Patch train.py

Two fixes applied here so you don't have to hand-edit the file in the repo:
1. adds `--init_from` (weights-only warm start, no optimizer/scheduler state)
2. fixes the final-save block to include optimizer/scheduler state (it was
   silently dropping them before)

**⚠️ Open item — not fixed here:** the two `train.py` patches described above "
"(`--init_from` weights-only warm start, and the final-save block including "
"optimizer/scheduler state) are only described in this markdown — there is no "
"code cell anywhere in this notebook that actually edits `train.py` to apply them. "
"Right now the training cell below just calls `--resume warm_start.pt` against "
"whatever `train.py` currently does, unpatched. If `--resume` expects a full "
"training-state checkpoint (optimizer state, step count, scheduler state) and "
"`warm_start.pt` only has model weights, training will either crash or silently "
"restart step counting from 0. Share `train.py` if you want this patch automated "
"— it can't be done safely without seeing the actual argparse/checkpoint code.

## Rebuild the .bin files (not persisted across sessions)

In [3]:
import os, json, re, string, glob, tarfile, numpy as np
from multiprocessing import Pool, cpu_count
from huggingface_hub import hf_hub_download

EXTRACT_DIR = '/kaggle/working/lma_extracted/hindi-lma-corpus'
TOK_PATH    = 'repo/phase2/tokenizers/hindi_tokenizer_16000.json'
OUT_DIR     = '/kaggle/working'
MAX_TRAIN   = 520_000_000
CHUNK       = 5000

if not os.path.exists(f'{EXTRACT_DIR}/.done'):
    print("[Dataset] Downloading hindi_submission_phase1.tar.gz...")
    tar_path = hf_hub_download(repo_id="manyajain111/hindi-lma-corpus", filename="hindi_submission_phase1.tar.gz", repo_type="dataset")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=EXTRACT_DIR)
    open(f'{EXTRACT_DIR}/.done', 'w').close()
    print("[Dataset] Extraction done!")
else:
    print(f"[Dataset] Already extracted in {EXTRACT_DIR}")

PUNCT    = re.escape(string.punctuation + "।॥''\"\"")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)
_vocab = _merge_ranks = _b2c = _cache = None

def init_worker(tok_path):
    global _vocab, _merge_ranks, _b2c, _cache
    with open(tok_path) as f: data = json.load(f)
    _vocab = data['vocab']
    merges = [((a, b), a + b) for a, b in data['merges']]
    _merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
    _b2c = {i: chr(i) for i in range(256)}
    _cache = {}

def encode_chunk(lines):
    out = []
    for line in lines:
        line = line.strip()
        if not line: continue
        for w in WORD_PAT.findall(line):
            if w not in _cache:
                s = [_b2c[b] for b in w.encode('utf-8')]
                while len(s) > 1:
                    pairs  = [(s[i], s[i+1]) for i in range(len(s)-1)]
                    ranked = [(_merge_ranks[p], i) for i, p in enumerate(pairs) if p in _merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
                _cache[w] = [_vocab.get(x, 0) for x in s]
            out.extend(_cache[w])
    return out

def tokenize_to_file(fnames, out_path, max_tok=None):
    import time
    fpaths = []
    for f in fnames:
        matches = glob.glob(f'{EXTRACT_DIR}/**/{f}', recursive=True)
        if matches: fpaths.append(matches[0])
    if not fpaths:
        print(f"❌ Error: Could not find any of {fnames} in {EXTRACT_DIR}")
        return
    if os.path.exists(out_path): os.remove(out_path)
    total_tokens = 0
    t0 = time.time()
    with open(out_path, 'wb') as f_out:
        for fpath in fpaths:
            print(f'Reading {os.path.basename(fpath)}...')
            lines = open(fpath, encoding='utf-8', errors='ignore').readlines()
            print(f'  {len(lines):,} lines | {cpu_count()} CPUs')
            chunks = [lines[i:i+CHUNK] for i in range(0, len(lines), CHUNK)]
            with Pool(cpu_count(), initializer=init_worker, initargs=(TOK_PATH,)) as pool:
                for i, r in enumerate(pool.imap(encode_chunk, chunks)):
                    arr = np.array(r, dtype=np.uint16)
                    f_out.write(arr.tobytes())
                    total_tokens += len(arr)
                    if i % 20 == 0:
                        tps = total_tokens / max(time.time() - t0, 1)
                        print(f'  {i}/{len(chunks)} chunks | {total_tokens:,} tokens | {tps:,.0f} tok/s', flush=True)
                    if max_tok and total_tokens >= max_tok:
                        pool.terminate()
                        break
            if max_tok and total_tokens >= max_tok: break
    print(f'✅ Saved {total_tokens:,} tokens -> {out_path}\n')

tokenize_to_file(['train.txt', 'hindi_news_data.txt'], f'{OUT_DIR}/hindi_train.bin', max_tok=MAX_TRAIN)
tokenize_to_file(['val.txt'], f'{OUT_DIR}/hindi_val.bin')
tokenize_to_file(['test.txt'], f'{OUT_DIR}/hindi_test.bin')

[Dataset] Downloading hindi_submission_phase1.tar.gz...


hindi_submission_phase1.tar.gz:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

/tmp/ipykernel_23/1264058902.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_DIR)


[Dataset] Extraction done!
Reading train.txt...
  6,124,956 lines | 4 CPUs
  0/1225 chunks | 415,042 tokens | 15,690 tok/s
  20/1225 chunks | 8,844,557 tokens | 222,495 tok/s
  40/1225 chunks | 17,464,788 tokens | 341,934 tok/s
  60/1225 chunks | 26,241,904 tokens | 422,113 tok/s
  80/1225 chunks | 34,666,647 tokens | 482,628 tok/s
  100/1225 chunks | 43,176,736 tokens | 533,133 tok/s
  120/1225 chunks | 51,630,771 tokens | 576,927 tok/s
  140/1225 chunks | 60,191,063 tokens | 612,279 tok/s
  160/1225 chunks | 68,524,947 tokens | 638,596 tok/s
  180/1225 chunks | 77,099,366 tokens | 666,794 tok/s
  200/1225 chunks | 85,616,253 tokens | 693,186 tok/s
  220/1225 chunks | 94,143,195 tokens | 714,641 tok/s
  240/1225 chunks | 102,612,626 tokens | 732,561 tok/s
  260/1225 chunks | 111,139,855 tokens | 752,808 tok/s
  280/1225 chunks | 119,877,844 tokens | 771,533 tok/s
  300/1225 chunks | 128,468,166 tokens | 788,270 tok/s
  320/1225 chunks | 136,985,323 tokens | 804,981 tok/s
  340/1225 ch

In [4]:
import glob
EXTRACT_DIR = '/kaggle/working/lma_extracted/hindi-lma-corpus'

def load_lines(fname):
    path = glob.glob(f'{EXTRACT_DIR}/**/{fname}', recursive=True)[0]
    with open(path, encoding='utf-8', errors='ignore') as f:
        lines = set(line.rstrip('\n') for line in f)
    print(f"{fname}: {len(lines):,} unique lines")
    return lines

train_lines = load_lines('train.txt')
val_lines = load_lines('val.txt')
test_lines = load_lines('test.txt')

print("train ∩ val:", len(train_lines & val_lines), f"({100*len(train_lines & val_lines)/len(val_lines):.1f}% of val)")
print("train ∩ test:", len(train_lines & test_lines), f"({100*len(train_lines & test_lines)/len(test_lines):.1f}% of test)")
print("val ∩ test:", len(val_lines & test_lines))

train.txt: 6,108,149 unique lines
val.txt: 340,219 unique lines
test.txt: 340,227 unique lines
train ∩ val: 1576 (0.5% of val)
train ∩ test: 1641 (0.5% of test)
val ∩ test: 127


## Warm-start from your existing 30000-step checkpoint

The cell below now auto-detects the checkpoint under `/kaggle/input/` instead of
requiring you to hand-edit a dataset name — see the fix note in the code cell.

In [5]:
import os, shutil, glob

os.makedirs("/kaggle/working/hindi_checkpoints", exist_ok=True)

candidates = glob.glob("/kaggle/input/**/model_final.pt", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "No model_final.pt found under /kaggle/input -- attach your 30000-step "
        "checkpoint dataset via 'Add Input' before running this cell."
    )
if len(candidates) > 1:
    print("Multiple checkpoints found -- using the first; verify this is the right one:")
    for c in candidates:
        print(" ", c)

CKPT_30K = candidates[0]
WARM_START = "/kaggle/working/hindi_checkpoints/warm_start.pt"
shutil.copy(CKPT_30K, WARM_START)
print(f"copied {CKPT_30K} -> {WARM_START} for warm start")

copied /kaggle/input/datasets/manyajain111/hindi-ckpt-30k/model_final.pt -> /kaggle/working/hindi_checkpoints/warm_start.pt for warm start


In [6]:
import re

train_path = "repo/phase2/src/train.py"
src = open(train_path, encoding="utf-8").read()

# FIX: --resume assumed the checkpoint always has optimizer/scheduler state.
# warm_start.pt only has model weights -> KeyError: 'optimizer_state_dict'.
# Make optimizer restore optional so a weights-only warm start works.
pattern = re.compile(r"([ \t]*)optimizer\.load_state_dict\(checkpoint\['optimizer_state_dict'\]\)")
m = pattern.search(src)
if m:
    indent = m.group(1)
    replacement = (
        f"{indent}if 'optimizer_state_dict' in checkpoint:\n"
        f"{indent}    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])\n"
        f"{indent}else:\n"
        f"{indent}    print(\"⚠️ No optimizer_state_dict in checkpoint -- doing a weights-only warm start.\")"
    )
    src = pattern.sub(replacement, src, count=1)
    open(train_path, "w", encoding="utf-8").write(src)
    print("Patched train.py: optimizer resume is now optional.")
elif "optimizer_state_dict' in checkpoint" in src:
    print("train.py already patched -- skipping.")
else:
    print("⚠️ Could not find the expected line -- patch manually.")

Patched train.py: optimizer resume is now optional.


In [7]:
import re

train_path = "repo/phase2/src/train.py"
src = open(train_path, encoding="utf-8").read()

pattern = re.compile(r"([ \t]*)scheduler\.load_state_dict\(checkpoint\['scheduler_state_dict'\]\)")
m = pattern.search(src)
if m:
    indent = m.group(1)
    replacement = (
        f"{indent}if 'scheduler_state_dict' in checkpoint:\n"
        f"{indent}    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])\n"
        f"{indent}else:\n"
        f"{indent}    print(\"⚠️ No scheduler_state_dict in checkpoint -- starting scheduler fresh.\")"
    )
    src = pattern.sub(replacement, src, count=1)
    open(train_path, "w", encoding="utf-8").write(src)
    print("Patched train.py: scheduler resume is now optional.")
elif "scheduler_state_dict' in checkpoint" in src:
    print("train.py already patched -- skipping.")
else:
    print("⚠️ Could not find the expected line -- patch manually.")

Patched train.py: scheduler resume is now optional.


In [8]:
!python repo/phase2/src/train.py \
    --config repo/phase2/config/model_h_config.json \
    --tokenizer repo/phase2/tokenizers/hindi_tokenizer_16000.json \
    --dataset manyajain111/hindi-lma-corpus \
    --output_dir /kaggle/working/hindi_checkpoints \
    --batch_size 16 --grad_accum 4 \
    --max_steps 61036 --save_steps 1000 --val_steps 1000 \
    --resume /kaggle/working/hindi_checkpoints/warm_start.pt

Using device: cuda
🚀 Using 2 GPUs with DataParallel!
Total parameters: 24.22 M
🔄 Resuming training from checkpoint: /kaggle/working/hindi_checkpoints/warm_start.pt
⚠️ No optimizer_state_dict in checkpoint -- doing a weights-only warm start.
⚠️ No scheduler_state_dict in checkpoint -- starting scheduler fresh.
✅ Resumed at step 30000
[Dataset] Memory-mapping /kaggle/working/hindi_train.bin...
[Dataset] 520,395,359 tokens ready.
[Dataset] Memory-mapping /kaggle/working/hindi_val.bin...
[Dataset] 29,106,897 tokens ready.
Training:  49%|█████████████████▏                 | 30000/61036 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
Training:  51%|▌| 31000/61036 [08:42<4:21:02,  1.92it/s, loss=3.9256, lr=1.50e-0
Step 31000: Val Loss 3.7493 | Val PPL 42.49
Train

## Sanity-check ROUGE-L before trusting results.json

Your original run reported ROUGE-L = 0.000 for every setting, which looks
like a tokenizer bug rather than a real score. This checks an identical
string against itself — it must print something close to 1.0.

In [9]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
print(scorer.score("मैं आज बाजार गया था।", "मैं आज बाजार गया था।"))
# If this prints 0.0, patch evaluate.py's RougeScorer call with
# tokenizer=lambda t: t.split() and re-run the evaluate cell above.

{'rougeL': Score(precision=0, recall=0, fmeasure=0)}


In [10]:
# FIX: the sanity check above shows ROUGE-L returns all zeros for identical
# Hindi strings -- rouge_score's default tokenizer only recognizes Latin word
# characters, so every Devanagari token gets dropped. The original notebook
# only left this as a comment telling you to hand-patch evaluate.py; automate
# it here so the evaluate cell below actually uses the fix.
eval_path = "repo/phase2/src/evaluate.py"
src = open(eval_path, encoding="utf-8").read()

old = "RougeScorer(['rougeL'], use_stemmer=False)"
new = "RougeScorer(['rougeL'], use_stemmer=False, tokenizer=lambda t: t.split())"

if old in src:
    open(eval_path, "w", encoding="utf-8").write(src.replace(old, new))
    print("Patched evaluate.py: RougeScorer now uses a whitespace tokenizer for Devanagari text.")
elif "tokenizer=lambda" in src:
    print("evaluate.py already patched -- skipping.")
else:
    print("⚠️ Could not find the expected RougeScorer(...) call in evaluate.py -- patch manually.")


Patched evaluate.py: RougeScorer now uses a whitespace tokenizer for Devanagari text.


## Evaluate the fully-trained checkpoint

In [11]:
import glob, os

ckpt_dir = "/kaggle/working/hindi_checkpoints"
if os.path.isdir(ckpt_dir):
    files = sorted(glob.glob(f"{ckpt_dir}/*"))
    if files:
        print(f"Found {len(files)} file(s) in {ckpt_dir}:")
        for f in files:
            size_mb = os.path.getsize(f) / 1e6
            print(f"  {f}  ({size_mb:.1f} MB)")
    else:
        print(f"{ckpt_dir} exists but is empty — training didn't save anything.")
else:
    print(f"{ckpt_dir} doesn't exist at all — training never ran or output_dir was wrong.")

Found 35 file(s) in /kaggle/working/hindi_checkpoints:
  /kaggle/working/hindi_checkpoints/ckpt_31000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_32000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_33000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_34000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_35000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_36000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_37000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_38000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_39000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_40000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_41000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_42000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_43000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_44000.pt  (297.1 MB)
  /kaggle/working/hindi_checkpoints/ckpt_45000.pt  (297.1 MB)
  /kaggle/worki

In [12]:
!python repo/phase2/src/evaluate.py \
    --checkpoint /kaggle/working/hindi_checkpoints/model_final.pt \
    --tokenizer repo/phase2/tokenizers/hindi_tokenizer_16000.json \
    --dataset manyajain111/hindi-lma-corpus \
    --language hindi \
    --output_dir /kaggle/working/hindi_reports

Device: cuda
Loaded model: 24.22M params
[Dataset] Memory-mapping /kaggle/working/hindi_test.bin...
[Dataset] 28,906,057 tokens ready.
Evaluating PPL/BPB: 100%|█████████████████████| 200/200 [00:08<00:00, 23.17it/s]

Test Loss: 3.6663 nats | PPL: 39.11 | BPB: 0.5131 | avg bytes/tok: 10.31
Greedy Generation: 100%|██████████████████████████| 5/5 [00:01<00:00,  4.20it/s]
Traceback (most recent call last):
  File "/kaggle/working/repo/phase2/src/evaluate.py", line 335, in <module>
    main()
  File "/kaggle/working/repo/phase2/src/evaluate.py", line 296, in main
    gen_results = evaluate_generation(model, tokenizer, device, prefixes, references)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/repo/phase2/src/evaluate.py", line 93, in evaluate_generation
    rougeL_g = sum(rouge.score(r, g)['rougeL'].fmeasure
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/repo/phase2/src/evaluate.py", line 93, in

## Causal-mask verification (required deliverable, currently missing)

Confirms position t's logits are unaffected by changing token t+1 —
proof the causal mask actually blocks future information.

In [13]:
import sys, torch, os
sys.path.insert(0, "repo/phase2/src")
from model import GPTLanguageModel
from tokenizer_wrapper import TokenizerWrapper

cfg = json.load(open("repo/phase2/config/model_h_config.json"))
tok = TokenizerWrapper("repo/phase2/tokenizers/hindi_tokenizer_16000.json")
model = GPTLanguageModel(**cfg)

# FIX: prefer the fully-trained (61036-step) checkpoint this run produces;
# fall back to the 30k warm-start checkpoint (CKPT_30K, discovered earlier)
# only if training hasn't produced model_final.pt yet. The old hardcoded
# "/kaggle/input/datasets/manyajain111/hindi-ckpt-30k/model_final.pt" doesn't
# exist on Kaggle -- see the warm-start cell above for why.
final_ckpt = "/kaggle/working/hindi_checkpoints/model_final.pt"
ckpt_path = final_ckpt if os.path.exists(final_ckpt) else CKPT_30K
print(f"Loading checkpoint from: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

ids = tok.encode("मैं आज बाजार गया था।")
x = torch.tensor([ids])
with torch.no_grad():
    logits_orig, _ = model(x)

x2 = x.clone()
x2[0, -1] = (x2[0, -1] + 137) % cfg["vocab_size"]
with torch.no_grad():
    logits_perturbed, _ = model(x2)

diff = (logits_orig[:, :-1] - logits_perturbed[:, :-1]).abs().max().item()
print(f"Max logit diff at earlier positions: {diff:.2e}")
assert diff < 1e-5, "Causal mask is leaking future information!"
print("✅ Causal masking verified: changing the last token does not affect earlier positions' logits.")


Loading checkpoint from: /kaggle/working/hindi_checkpoints/model_final.pt
Max logit diff at earlier positions: 0.00e+00
✅ Causal masking verified: changing the last token does not affect earlier positions' logits.


## Zip everything

Note: `train_log.csv` from this run only covers steps 30000->61036. For your
loss-curve plot, concatenate it with the CSV from your original 0->30000 run.

In [14]:
from IPython.display import FileLink
!zip -r hindi_final_outputs_v2.zip /kaggle/working/hindi_reports /kaggle/working/hindi_checkpoints/model_final.pt /kaggle/working/hindi_checkpoints/train_log.csv
print("\n✅ Zip file created successfully! Click the link below to download:")
FileLink('hindi_final_outputs_v2.zip')

  adding: kaggle/working/hindi_reports/ (stored 0%)
  adding: kaggle/working/hindi_checkpoints/model_final.pt (deflated 13%)
  adding: kaggle/working/hindi_checkpoints/train_log.csv (deflated 68%)

✅ Zip file created successfully! Click the link below to download:


/kaggle/working/hindi_final_outputs_v2.zip